In [1]:
import os
import torch

os.environ["WANDB_DISABLED"] = "true"

YOLOV5_DIR = "/content/yolov5"

if not os.path.exists(YOLOV5_DIR):
    !git clone https://github.com/ultralytics/yolov5 {YOLOV5_DIR}
%cd yolov5

!pip install -q -r {YOLOV5_DIR}/requirements.txt
!pip install -q deep-sort-realtime==1.3.2

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Cloning into '/content/yolov5'...
remote: Enumerating objects: 17903, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 17903 (delta 41), reused 9 (delta 9), pack-reused 17850 (from 5)
Receiving objects: 100% (17903/17903), 17.06 MiB | 25.69 MiB/s, done.
Resolving deltas: 100% (12195/12195), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 55.6 MB/s eta 0:00:00
PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4


In [2]:
%cd /content
import os

DATASET_DIR = "/content/datasets/visdrone"
os.makedirs(DATASET_DIR, exist_ok=True)

train_zip = os.path.join(DATASET_DIR, "train.zip")
val_zip = os.path.join(DATASET_DIR, "val.zip")

if not os.path.exists(train_zip):
    !wget -q -O {train_zip} https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip

if not os.path.exists(val_zip):
    !wget -q -O {val_zip} https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip

train_dir = os.path.join(DATASET_DIR, "VisDrone2019-DET-train")
val_dir = os.path.join(DATASET_DIR, "VisDrone2019-DET-val")

if not os.path.exists(train_dir):
    !unzip -q {train_zip} -d {DATASET_DIR}

if not os.path.exists(val_dir):
    !unzip -q {val_zip} -d {DATASET_DIR}

assert os.path.exists(os.path.join(train_dir, "images"))
assert os.path.exists(os.path.join(val_dir, "images"))

print("VisDrone DET dataset ready")

/content
VisDrone DET dataset ready


In [3]:
import os
from PIL import Image

def convert_visdrone(base_path):
    for split in ['train', 'val']:

        img_dir = f"{base_path}/VisDrone2019-DET-{split}/images"
        ann_dir = f"{base_path}/VisDrone2019-DET-{split}/annotations"
        label_dir = f"{base_path}/VisDrone2019-DET-{split}/labels"

        os.makedirs(label_dir, exist_ok=True)

        for file in os.listdir(ann_dir):
            if not file.endswith('.txt'):
                continue

            img_path = os.path.join(img_dir, file.replace('.txt', '.jpg'))
            if not os.path.exists(img_path):
                print(f"Missing image: {img_path}")
                continue

            w, h = Image.open(img_path).size
            yolo_lines = []

            with open(os.path.join(ann_dir, file), 'r') as f:
                for line in f:
                    parts = line.strip().split(',')
                    if len(parts) < 8:
                        continue

                    x, y, bw, bh = map(float, parts[:4])
                    cls_id = int(parts[5])

                    if cls_id in [0, 11]:
                        continue

                    cls_id -= 1

                    xc = (x + bw / 2) / w
                    yc = (y + bh / 2) / h
                    bw /= w
                    bh /= h

                    if bw <= 0 or bh <= 0:
                        continue

                    yolo_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

            with open(os.path.join(label_dir, file), 'w') as out:
                out.write('\n'.join(yolo_lines))

convert_visdrone('/content/datasets/visdrone')

In [4]:
%%writefile /content/yolov5/visdrone.yaml

path: /content/datasets/visdrone

train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-val/images

# VisDrone classes after removing:
# 0 = ignored regions
# 11 = others
nc: 10

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor

Writing /content/yolov5/visdrone.yaml


In [5]:
%cd /content/yolov5

!python train.py \
  --img 640 \
  --batch 8 \
  --epochs 20 \
  --data visdrone.yaml \
  --weights yolov5s.pt \
  --cache \
  --name visdrone_640

Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
      16/19         2G    0.09689     0.1597    0.02737        555        640:  93% 756/809 [02:18<00:08,  6.00it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      16/19         2G    0.09689     0.1597    0.02737        657        640:  94% 757/809 [02:19<00:08,  6.05it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      16/19         2G    0.09688     0.1597    0.02737        535        640:  94% 758/809 [02:19<00:09,  5.54it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(

In [6]:
!python val.py \
  --weights runs/train/visdrone_640/weights/best.pt \
  --data visdrone.yaml \
  --img 640

val: data=visdrone.yaml, weights=['runs/train/visdrone_640/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7037095 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/datasets/visdrone/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% 548/548 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 18/18 [00:21<00:00,  1.17s/it]
                   all        548      38759      0.381      0.297      0.277      0.144
            pedestrian        548       8844      0.401      0.376       0.36      0.148
      

In [7]:
!python train.py \
  --img 1280 \
  --batch 4 \
  --epochs 10 \
  --patience 3 \
  --data visdrone.yaml \
  --weights runs/train/visdrone_640/weights/best.pt \
  --name visdrone_1280 \
  --workers 2

Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
        8/9      3.66G    0.07817     0.2237    0.02343        236       1280:  46% 751/1618 [05:37<05:00,  2.89it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
        8/9      3.66G    0.07815     0.2237    0.02344        232       1280:  46% 752/1618 [05:37<05:20,  2.70it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
        8/9      3.66G    0.07814     0.2237    0.02344        258       1280:  47% 753/1618 [05:38<04:52,  2.95it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autoca

In [8]:
!python val.py \
  --weights runs/train/visdrone_1280/weights/best.pt \
  --data visdrone.yaml \
  --img 1280

val: data=visdrone.yaml, weights=['runs/train/visdrone_1280/weights/best.pt'], batch_size=32, imgsz=1280, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7037095 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/datasets/visdrone/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% 548/548 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 18/18 [00:32<00:00,  1.78s/it]
                   all        548      38759      0.476      0.434      0.418      0.234
            pedestrian        548       8844      0.546      0.564      0.573      0.266
    

In [9]:
!wget --no-check-certificate -O /content/drone_test.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4"

--2026-04-23 23:34:17--  https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4 [following]
--2026-04-23 23:34:18--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6031199 (5.8M) [application/octet-stream]
Saving to: ‘/content/drone_test.mp4’

/content/drone_test 100%[===================>]   5.75M  --.-KB/s    in 0.08s   

2026-04-23 23

In [10]:
import warnings
warnings.filterwarnings("ignore")

import os
import cv2
import torch
import time
from deep_sort_realtime.deepsort_tracker import DeepSort

MODEL_PATH = '/content/yolov5/runs/train/visdrone_1280/weights/best.pt'
VIDEO_PATH = '/content/drone_test.mp4'
OUTPUT_PATH = '/content/final_output.mp4'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

model = torch.hub.load(
    '/content/yolov5',
    'custom',
    path=MODEL_PATH,
    source='local'
)

model.to(device)
model.eval()

model.conf = 0.25
model.iou = 0.45

tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_cosine_distance=0.4
)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {VIDEO_PATH}")

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video resolution:", width, "x", height)
print("FPS:", fps)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

frame_count = 0
total_tracks = 0
start_time = time.time()

with torch.no_grad():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, size=1280)
        detections = results.xyxy[0].cpu().numpy()

        ds_input = []

        for x1, y1, x2, y2, conf, cls in detections:
            if conf < 0.3:
                continue

            ds_input.append((
                [int(x1), int(y1), int(x2 - x1), int(y2 - y1)],
                float(conf),
                int(cls)
            ))

        tracks = tracker.update_tracks(ds_input, frame=frame)

        visible_tracks = 0

        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = int(track.track_id)

            l, t, r, b = map(int, track.to_ltrb())

            cls_id = track.get_det_class()
            if cls_id is not None and int(cls_id) in model.names:
              label = model.names[int(cls_id)]
            else:
              label = "Object"

            color = (
                int((track_id * 37) % 255),
                int((track_id * 17) % 255),
                int((track_id * 29) % 255)
            )

            cv2.rectangle(frame, (l, t), (r, b), color, 2)

            cv2.putText(
                frame,
                f"ID:{track_id} {label}",
                (l, max(t - 10, 0)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            visible_tracks += 1

        out.write(frame)

        frame_count += 1
        total_tracks += visible_tracks

        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames...")

cap.release()
out.release()

elapsed = time.time() - start_time
avg_fps = frame_count / elapsed if elapsed > 0 else 0
avg_tracks = total_tracks / frame_count if frame_count > 0 else 0

print(f"Saved tracked video to: {OUTPUT_PATH}")
print(f"Average FPS: {avg_fps:.2f}")
print(f"Average tracked objects/frame: {avg_tracks:.2f}")

Using device: cuda


YOLOv5 🚀 v7.0-472-g7ca403a3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7037095 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Video resolution: 768 x 432
FPS: 12
Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processed 450 frames...
Processed 500 frames...
Processed 550 frames...
Processed 600 frames...
Saved tracked video to: /content/final_output.mp4
Average FPS: 29.33
Average tracked objects/frame: 0.96


In [11]:
import os
from google.colab import files

file_path = "/content/final_output.mp4"

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"Downloading final_output.mp4 ({size_mb:.2f} MB)")
    files.download(file_path)
else:
    print("final_output.mp4 not found.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import pandas as pd

df_640 = pd.read_csv("/content/yolov5/runs/train/visdrone_640/results.csv")
df_1280 = pd.read_csv("/content/yolov5/runs/train/visdrone_1280/results.csv")

df_640.columns = df_640.columns.str.strip()
df_1280.columns = df_1280.columns.str.strip()

map50_640 = df_640["metrics/mAP_0.5"].iloc[-1]
map50_1280 = df_1280["metrics/mAP_0.5"].iloc[-1]

improvement = ((map50_1280 - map50_640) / map50_640) * 100
tracking_fps = round(avg_fps, 2) if 'avg_fps' in globals() else None

results_df = pd.DataFrame([{
    "640_model_mAP50": round(map50_640, 4),
    "1280_model_mAP50": round(map50_1280, 4),
    "mAP_Improvement_%": round(improvement, 2),
    "Tracking_FPS": tracking_fps
}])

results_df

,640_model_mAP50,1280_model_mAP50,mAP_Improvement_%,Tracking_FPS
0,0.2762,0.4172,51.09,29.33
